In [ ]:
# core_glaucogate.py
import os
import math
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
import torchvision.models as tv_models
import numpy as np
import pandas as pd

try:
    import timm
    HAS_TIMM = True
except ImportError:
    HAS_TIMM = False

# SYNTHETIC DEGRADATION & SALIENCY PRIOR
def compute_onh_saliency_prior(img_rgb_tensor, grid_size=32):
    red_green = 0.5 * img_rgb_tensor[0] + 0.5 * img_rgb_tensor[1]
    prior = F.adaptive_avg_pool2d(red_green.unsqueeze(0).unsqueeze(0), (grid_size, grid_size)).squeeze(0).squeeze(0)
    yy, xx = torch.meshgrid(torch.linspace(-1, 1, grid_size), torch.linspace(-1, 1, grid_size), indexing="ij")
    fov_weight = (xx**2 + yy**2 < 0.72).float()
    prior = prior * fov_weight + 1e-6
    return prior / prior.sum()

def apply_synthetic_degradation(img_np, deg_type="random", severity=0.5, disc_center=None):
    h, w, _ = img_np.shape
    gt_quality_map = np.ones((h, w), dtype=np.float32)
    mask_omega = np.zeros((h, w), dtype=np.float32)
    if severity <= 0.0 or deg_type == "none":
        return img_np, gt_quality_map, mask_omega

    if deg_type == "random":
        deg_type = np.random.choice(["blur", "glare", "dark", "haze", "disc_corruption"])

    if deg_type == "disc_corruption":
        if disc_center is not None and not np.isnan(disc_center[0]):
            cx = int((disc_center[0] + 1.0) * 0.5 * w)
            cy = int((disc_center[1] + 1.0) * 0.5 * h)
        else:
            cx, cy = int(0.5 * w), int(0.5 * h)
        radius = int(0.18 * w * (0.65 + 0.35 * severity))
    else:
        cx = np.random.randint(int(0.25 * w), int(0.75 * w))
        cy = np.random.randint(int(0.25 * h), int(0.75 * h))
        radius = int(w * (0.15 + 0.25 * severity))

    cv2.circle(mask_omega, (cx, cy), max(5, radius), 1.0, -1)
    smooth_omega = cv2.GaussianBlur(mask_omega, (51, 51), max(1.0, radius * 0.4))
    smooth_omega = np.clip(smooth_omega, 0.0, 1.0)
    mask_3c = smooth_omega[:, :, None]

    degraded = img_np.astype(np.float32).copy()
    if deg_type in ["blur", "disc_corruption"]:
        ksize = int(15 + 48 * severity) | 1
        corrupted = cv2.GaussianBlur(degraded, (ksize, ksize), 8 + 18 * severity)
    elif deg_type == "glare":
        corrupted = np.clip(degraded + 255.0 * (0.5 + 0.5 * severity), 0, 255)
    elif deg_type == "dark":
        corrupted = degraded * max(0.05, 1.0 - 0.92 * severity)
    elif deg_type == "haze":
        corrupted = degraded * (1.0 - 0.75 * severity) + 215.0 * (0.75 * severity)
    else:
        corrupted = degraded

    out_img = (degraded * (1.0 - mask_3c) + corrupted * mask_3c).astype(np.uint8)
    gt_quality_map = np.clip(1.0 - smooth_omega * severity * 0.95, 0.02, 1.0)
    return out_img, gt_quality_map, (mask_omega > 0.5).astype(np.float32)

class FundusDataset(Dataset):
    def __init__(self, csv_path, split=None, global_size=512, highres_size=768,
                 is_train=False, deg_prob=0.25, fixed_deg_type="none", fixed_severity=0.0):
        df = pd.read_csv(csv_path)
        if split is not None and "split" in df.columns:
            df = df[df["split"] == split].reset_index(drop=True)
        self.df = df
        self.global_size = global_size
        self.highres_size = highres_size
        self.is_train = is_train
        self.deg_prob = deg_prob
        self.fixed_deg_type = fixed_deg_type
        self.fixed_severity = fixed_severity

    def set_deg_prob(self, new_prob):
        self.deg_prob = float(new_prob)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_bgr = cv2.imread(str(row["image_path"]))
        if img_bgr is None:
            img_rgb = np.zeros((self.highres_size, self.highres_size, 3), dtype=np.uint8)
        else:
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        if self.is_train and np.random.rand() < 0.5:
            img_rgb = cv2.flip(img_rgb, 1)

        disc_loc = (row.get("od_cx", np.nan), row.get("od_cy", np.nan))
        if self.fixed_deg_type != "none" and self.fixed_severity > 0:
            img_rgb, gt_qmap, omega_mask = apply_synthetic_degradation(
                img_rgb, deg_type=self.fixed_deg_type, severity=self.fixed_severity, disc_center=disc_loc
            )
        elif self.is_train and np.random.rand() < self.deg_prob:
            sev = np.random.uniform(0.35, 0.85)
            img_rgb, gt_qmap, omega_mask = apply_synthetic_degradation(
                img_rgb, deg_type="random", severity=sev, disc_center=disc_loc
            )
        else:
            gt_qmap = np.ones((img_rgb.shape[0], img_rgb.shape[1]), dtype=np.float32)
            omega_mask = np.zeros((img_rgb.shape[0], img_rgb.shape[1]), dtype=np.float32)

        img_highres = cv2.resize(img_rgb, (self.highres_size, self.highres_size))
        img_global = cv2.resize(img_rgb, (self.global_size, self.global_size))
        gt_qmap_small = cv2.resize(gt_qmap, (16, 16), interpolation=cv2.INTER_AREA)
        omega_small = cv2.resize(omega_mask, (16, 16), interpolation=cv2.INTER_NEAREST)

        highres_t = torch.from_numpy(img_highres).permute(2, 0, 1).float() / 255.0
        global_t = torch.from_numpy(img_global).permute(2, 0, 1).float() / 255.0
        saliency_prior = compute_onh_saliency_prior(global_t, grid_size=32)

        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        highres_t = (highres_t - mean) / std
        global_t = (global_t - mean) / std

        return {
            "image_id": str(row["image_id"]),
            "patient_id": str(row.get("patient_id", row["image_id"])),
            "global_img": global_t,
            "highres_img": highres_t,
            "label": torch.tensor(float(row["label"]), dtype=torch.float32),
            "pseudo_domain": torch.tensor(int(row.get("pseudo_domain", 0)), dtype=torch.long),
            "gt_qmap": torch.from_numpy(gt_qmap_small).unsqueeze(0).float(),
            "omega_mask": torch.from_numpy(omega_small).unsqueeze(0).float(),
            "saliency_prior": saliency_prior,
            "od_gt": torch.tensor([
                float(row.get("od_cx", np.nan)),
                float(row.get("od_cy", np.nan)),
                float(row.get("od_r", np.nan))
            ], dtype=torch.float32)
        }
# CONVNEXT BACKBONE & GLAUCOGATE MODULES
class ConvNeXtFeatureExtractor(nn.Module):
    def __init__(self, backbone_name="convnext_base.fb_in22k_ft_in1k", pretrained=True, drop_rate=0.2):
        super().__init__()
        self.use_timm = False
        is_tiny = ("tiny" in backbone_name.lower())

        try:
            if is_tiny:
                weights = tv_models.ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None
                m = tv_models.convnext_tiny(weights=weights, stochastic_depth_prob=drop_rate)
                self.out_channels = 768
            else:
                weights = tv_models.ConvNeXt_Base_Weights.DEFAULT if pretrained else None
                m = tv_models.convnext_base(weights=weights, stochastic_depth_prob=drop_rate)
                self.out_channels = 1024
            self.net = m.features
            return
        except Exception:
            pass

        if HAS_TIMM:
            try:
                self.net = timm.create_model(backbone_name, pretrained=pretrained, features_only=True, drop_rate=drop_rate)
                self.out_channels = self.net.feature_info.channels()[-1]
                self.use_timm = True
                return
            except Exception:
                pass

        if is_tiny:
            m = tv_models.convnext_tiny(weights=None, stochastic_depth_prob=drop_rate)
            self.out_channels = 768
        else:
            m = tv_models.convnext_base(weights=None, stochastic_depth_prob=drop_rate)
            self.out_channels = 1024
        self.net = m.features

    def forward(self, x):
        if self.use_timm:
            return self.net(x)[-1]
        return self.net(x)

class PolarISNTQuadrantAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.quadrant_mlp = nn.Sequential(
            nn.Linear(channels * 4, channels // 2),
            nn.LayerNorm(channels // 2),
            nn.GELU(),
            nn.Linear(channels // 2, 4),
            nn.Softmax(dim=-1)
        )
        self.radial_conv = nn.Conv2d(channels, channels, kernel_size=3, padding=1, groups=channels)
        self.proj = nn.Conv2d(channels, channels, kernel_size=1)
        self.gamma = nn.Parameter(torch.tensor(0.1))

    def forward(self, x):
        B, C, H, W = x.shape
        device = x.device
        yy, xx = torch.meshgrid(
            torch.linspace(-1, 1, H, device=device, dtype=torch.float32),
            torch.linspace(-1, 1, W, device=device, dtype=torch.float32),
            indexing="ij"
        )
        theta = torch.atan2(yy, xx)
        q_inf = ((theta >= math.pi / 4) & (theta < 3 * math.pi / 4)).to(x.dtype)
        q_sup = ((theta >= -3 * math.pi / 4) & (theta < -math.pi / 4)).to(x.dtype)
        q_temp = ((theta >= -math.pi / 4) & (theta < math.pi / 4)).to(x.dtype)
        q_nas = ((theta >= 3 * math.pi / 4) | (theta < -3 * math.pi / 4)).to(x.dtype)

        masks = torch.stack([q_inf, q_sup, q_nas, q_temp], dim=0).unsqueeze(0).unsqueeze(2)
        x_exp = x.unsqueeze(1)
        quad_pooled = (x_exp * masks).sum(dim=(-1, -2)) / (masks.sum(dim=(-1, -2)) + 1e-6)

        quad_weights = self.quadrant_mlp(quad_pooled.reshape(B, -1))
        spatial_weight_map = (quad_weights.view(B, 4, 1, 1, 1) * masks).sum(dim=1)

        delta = self.proj(x * spatial_weight_map + self.radial_conv(x))
        return x + self.gamma * delta

class MultiScaleRNFLModule(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(channels, channels, kernel_size=3, padding=d, dilation=d, groups=channels, bias=False),
                nn.BatchNorm2d(channels),
                nn.GELU()
            ) for d in [1, 2, 4, 6]
        ])
        self.fuse = nn.Sequential(
            nn.Conv2d(channels * 4, channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.GELU()
        )
        self.gamma = nn.Parameter(torch.tensor(0.1))

    def forward(self, x):
        return x + self.gamma * self.fuse(torch.cat([b(x) for b in self.branches], dim=1))

class WeaklySupervisedONHLocalizer(nn.Module):
    def __init__(self, s_min=0.20, s_max=0.45, default_s=0.30):
        super().__init__()
        self.s_min = s_min
        self.s_max = s_max
        self.default_s = default_s
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 3)
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.constant_(self.net[-1].bias, 0.0)

    def forward(self, x_global, highres_img, local_size=384, force_center_crop=False, oracle_coords=None):
        B = x_global.shape[0]
        with torch.amp.autocast("cuda" if x_global.is_cuda else "cpu", enabled=False):
            x_down = F.interpolate(x_global.float(), size=(128, 128), mode="bilinear", align_corners=False)
            raw = self.net.float()(x_down)

            cx = torch.tanh(raw[:, 0]) * 0.70
            cy = torch.tanh(raw[:, 1]) * 0.70
            s = self.s_min + (self.s_max - self.s_min) * torch.sigmoid(raw[:, 2])

            failed_mask = torch.isnan(cx) | torch.isnan(cy) | (cx.abs() > 0.80) | (cy.abs() > 0.80)
            cx = torch.where(failed_mask, torch.zeros_like(cx), cx)
            cy = torch.where(failed_mask, torch.zeros_like(cy), cy)
            s = torch.where(failed_mask, torch.full_like(s, self.default_s), s)

            if force_center_crop:
                cx = torch.zeros_like(cx)
                cy = torch.zeros_like(cy)
                s = torch.full_like(s, self.default_s)
            elif oracle_coords is not None:
                valid_oracle = ~torch.isnan(oracle_coords[:, 0])
                cx = torch.where(valid_oracle, oracle_coords[:, 0].float(), cx)
                cy = torch.where(valid_oracle, oracle_coords[:, 1].float(), cy)
                s = torch.where(
                    valid_oracle & (oracle_coords[:, 2] > 0),
                    torch.clamp(oracle_coords[:, 2].float() * 1.8, self.s_min, self.s_max), s
                )

            zeros = torch.zeros_like(s)
            theta_fwd = torch.stack([
                torch.stack([s, zeros, cx], dim=-1),
                torch.stack([zeros, s, cy], dim=-1)
            ], dim=1)

            grid_fwd = F.affine_grid(theta_fwd, size=(B, 3, local_size, local_size), align_corners=False)
            crop_img = F.grid_sample(highres_img.float(), grid_fwd, mode="bilinear", padding_mode="zeros", align_corners=False)
            stn_tensor = torch.stack([cx, cy, s], dim=1)
        return crop_img.to(x_global.dtype).contiguous(), stn_tensor

    @staticmethod
    def inverse_warp_local_to_global(f_loc, stn_tensor, target_h, target_w):
        if isinstance(stn_tensor, (tuple, list)):
            cx, cy, s = stn_tensor[0].float(), stn_tensor[1].float(), stn_tensor[2].float()
        else:
            cx, cy, s = stn_tensor[:, 0].float(), stn_tensor[:, 1].float(), stn_tensor[:, 2].float()
        B, C, _, _ = f_loc.shape
        with torch.amp.autocast("cuda" if f_loc.is_cuda else "cpu", enabled=False):
            inv_s = 1.0 / (s + 1e-6)
            zeros = torch.zeros_like(inv_s)
            theta_inv = torch.stack([
                torch.stack([inv_s, zeros, -cx * inv_s], dim=-1),
                torch.stack([zeros, inv_s, -cy * inv_s], dim=-1)
            ], dim=1)
            grid_inv = F.affine_grid(theta_inv, size=(B, C, target_h, target_w), align_corners=False)
            warped = F.grid_sample(f_loc.float(), grid_inv, mode="bilinear", padding_mode="zeros", align_corners=False)
        return warped.to(f_loc.dtype).contiguous()

class QualityEstimationBranch(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.decoder = nn.Sequential(
            nn.Conv2d(in_channels, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256), nn.GELU(),
            nn.Conv2d(256, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, 1, kernel_size=1)
        )
        nn.init.constant_(self.decoder[-1].bias, 2.0)

    def forward(self, f_glob):
        raw_logit = self.decoder(f_glob)
        return 10.0 * torch.tanh(raw_logit.float() / 10.0)

class QCAFModule(nn.Module):
    def __init__(self, channels, fusion_mode="multiplicative"):
        super().__init__()
        self.fusion_mode = fusion_mode
        self.norm_loc = nn.LayerNorm(channels)
        self.norm_glob = nn.LayerNorm(channels)
        self.attn_conv = nn.Conv2d(channels * 2, channels, kernel_size=1)
        self.alpha = nn.Parameter(torch.ones(1, channels, 1, 1))
        self.beta = nn.Parameter(torch.zeros(1, channels, 1, 1))
        if fusion_mode == "concat":
            self.concat_proj = nn.Sequential(
                nn.Conv2d(channels * 2, channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(channels),
                nn.GELU()
            )

    def forward(self, f_loc_aligned, f_glob, m_q_prob):
        m_q_resized = F.interpolate(m_q_prob.float(), size=f_glob.shape[-2:], mode="bilinear", align_corners=False).to(f_glob.dtype)

        if self.fusion_mode == "concat":
            gate = torch.full_like(f_glob, 0.5)
            f_fused = self.concat_proj(torch.cat([f_loc_aligned, f_glob], dim=1))
            return f_fused, gate

        a_s = torch.sigmoid(self.attn_conv(torch.cat([f_loc_aligned, f_glob], dim=1)))

        if self.fusion_mode == "ungated_attention":
            gate = a_s
        elif self.fusion_mode == "original_additive":
            gate = torch.sigmoid(self.alpha * (a_s * m_q_resized) + self.beta)
        elif self.fusion_mode == "multiplicative":
            gate = m_q_resized * torch.sigmoid(self.alpha * a_s + self.beta)
        else:
            raise ValueError(f"Unknown fusion_mode: {self.fusion_mode}")

        f_fused = gate * f_loc_aligned + (1.0 - gate) * f_glob
        return f_fused, gate

class ConvNeXtBaseModel(nn.Module):
    def __init__(self, backbone_name="convnext_base.fb_in22k_ft_in1k", pretrained=True, drop_rate=0.2):
        super().__init__()
        self.encoder = ConvNeXtFeatureExtractor(backbone_name, pretrained=pretrained, drop_rate=drop_rate)
        self.norm = nn.LayerNorm(self.encoder.out_channels)
        self.head = nn.Linear(self.encoder.out_channels, 1)

    def forward(self, x_glob):
        if isinstance(x_glob, dict):
            x_glob = x_glob["global_img"]
        f_glob = self.encoder(x_glob)
        pooled = self.norm(F.adaptive_avg_pool2d(f_glob, 1).flatten(1).float())
        raw_logit = self.head(pooled).squeeze(-1)
        logit = 20.0 * torch.tanh(raw_logit / 20.0)
        return {
            "logit": logit,
            "prob": torch.sigmoid(logit),
            "selection_score": torch.ones_like(logit)
        }

class ConvNeXtProGlaucoGate(nn.Module):
    def __init__(self, backbone_name="convnext_base.fb_in22k_ft_in1k", pretrained=True,
                 fusion_mode="multiplicative", ablation_mode="full", drop_rate=0.2):
        super().__init__()
        self.ablation_mode = ablation_mode
        self.localizer = WeaklySupervisedONHLocalizer()

        self.global_encoder = ConvNeXtFeatureExtractor(backbone_name, pretrained=pretrained, drop_rate=drop_rate)
        self.local_encoder = ConvNeXtFeatureExtractor(backbone_name, pretrained=pretrained, drop_rate=drop_rate)
        in_ch = self.global_encoder.out_channels

        self.polar_isnt = PolarISNTQuadrantAttention(in_ch)
        self.ms_rnfl = MultiScaleRNFLModule(in_ch)
        self.quality_branch = QualityEstimationBranch(in_ch)
        self.qcaf = QCAFModule(in_ch, fusion_mode=fusion_mode)

        self.feat_norm = nn.LayerNorm(in_ch)
        self.dropout = nn.Dropout(drop_rate)
        self.cls_head = nn.Linear(in_ch, 1)

        self.sel_head = nn.Linear(in_ch + 1, 1)
        nn.init.zeros_(self.sel_head.weight)
        nn.init.constant_(self.sel_head.bias, 2.44)

    def forward(self, x_glob, x_highres=None, force_center_crop=False, oracle_coords=None):
        if isinstance(x_glob, dict):
            batch = x_glob
            x_glob = batch["global_img"]
            x_highres = batch["highres_img"]

        f_glob_raw = self.global_encoder(x_glob)
        f_glob = self.ms_rnfl(f_glob_raw)

        m_q_logit = self.quality_branch(f_glob_raw)
        m_q_prob = torch.sigmoid(m_q_logit).to(f_glob.dtype)

        crop_img, stn_tensor = self.localizer(
            x_glob, x_highres, local_size=384,
            force_center_crop=force_center_crop, oracle_coords=oracle_coords
        )
        f_loc_raw = self.local_encoder(crop_img)
        f_loc_isnt = self.polar_isnt(f_loc_raw)

        f_loc_aligned = self.localizer.inverse_warp_local_to_global(
            f_loc_isnt, stn_tensor, target_h=f_glob.shape[-2], target_w=f_glob.shape[-1]
        )

        if self.ablation_mode == "global_only":
            f_fused = f_glob
            gate = torch.zeros_like(f_glob)
        elif self.ablation_mode == "local_only":
            f_fused = F.adaptive_avg_pool2d(f_loc_isnt, f_glob.shape[-2:])
            gate = torch.ones_like(f_glob)
        else:
            f_fused, gate = self.qcaf(f_loc_aligned, f_glob, m_q_prob)

        pooled_feat = self.dropout(self.feat_norm(F.adaptive_avg_pool2d(f_fused, 1).flatten(1).float()))
        pooled_mq = F.adaptive_avg_pool2d(m_q_prob.float(), 1).flatten(1)

        raw_logit = self.cls_head(pooled_feat).squeeze(-1)
        logit = 20.0 * torch.tanh(raw_logit / 20.0)

        sel_input = torch.cat([pooled_feat.detach(), pooled_mq], dim=1)
        sel_score = torch.sigmoid(self.sel_head(sel_input).squeeze(-1))

        return {
            "logit": logit,
            "prob": torch.sigmoid(logit),
            "selection_score": sel_score,
            "m_q": m_q_prob,
            "m_q_logit": m_q_logit,
            "gate": gate,
            "stn_params": stn_tensor
        }

# LOSS SUITE (Memory-Bank pAUC + Group-DRO)
class MemoryBankPartialAUCLoss(nn.Module):
    def __init__(self, margin=1.0, fpr_max=0.10, queue_size=512, use_memory=True):
        super().__init__()
        self.margin = margin
        self.fpr_max = fpr_max
        self.queue_size = queue_size
        self.use_memory = use_memory
        self.register_buffer("neg_queue", torch.full((queue_size,), -3.0))
        self.register_buffer("queue_ptr", torch.zeros(1, dtype=torch.long))
        self.register_buffer("queue_filled", torch.zeros(1, dtype=torch.bool))

    @torch.no_grad()
    def _update_queue(self, neg_logits):
        n = neg_logits.shape[0]
        if n == 0:
            return
        clean_negs = torch.clamp(neg_logits.detach().float(), -20.0, 20.0)
        ptr = int(self.queue_ptr.item())
        if ptr + n <= self.queue_size:
            self.neg_queue[ptr:ptr + n] = clean_negs
        else:
            rem = self.queue_size - ptr
            self.neg_queue[ptr:] = clean_negs[:rem]
            self.neg_queue[:n - rem] = clean_negs[rem:]
            self.queue_filled[0] = True
        self.queue_ptr[0] = (ptr + n) % self.queue_size

    def forward(self, logits, targets):
        logits = logits.float()
        targets = targets.float()
        pos_mask = (targets > 0.5)
        neg_mask = ~pos_mask
        pos_logits = logits[pos_mask]
        neg_logits = logits[neg_mask]

        if pos_logits.numel() == 0 or neg_logits.numel() == 0:
            return F.binary_cross_entropy_with_logits(logits, targets)

        if self.use_memory:
            active_q = self.neg_queue if self.queue_filled[0] else self.neg_queue[:max(1, int(self.queue_ptr.item()))]
            all_negs = torch.cat([neg_logits, active_q.to(logits.device).float()], dim=0)
        else:
            all_negs = neg_logits

        k_hard = max(1, int(all_negs.numel() * self.fpr_max))
        hard_negs, _ = torch.topk(all_negs, k=k_hard, largest=True)

        viol_pos = F.relu(hard_negs.unsqueeze(0).detach() - pos_logits.unsqueeze(1) + self.margin)
        loss_pos = F.smooth_l1_loss(viol_pos, torch.zeros_like(viol_pos), beta=0.5)

        k_batch_hard = max(1, int(neg_logits.numel() * self.fpr_max))
        batch_hard_negs, _ = torch.topk(neg_logits, k=k_batch_hard, largest=True)
        viol_neg = F.relu(batch_hard_negs.unsqueeze(0) - pos_logits.detach().unsqueeze(1) + self.margin)
        loss_neg = F.smooth_l1_loss(viol_neg, torch.zeros_like(viol_neg), beta=0.5)

        if self.training and self.use_memory:
            self._update_queue(neg_logits)
        return 0.5 * (loss_pos + loss_neg)

class PseudoDomainGroupDRO(nn.Module):
    def __init__(self, n_domains=10, step_size=0.01):
        super().__init__()
        self.n_domains = n_domains
        self.step_size = step_size
        self.register_buffer("q_weights", torch.ones(n_domains) / n_domains)

    def forward(self, per_sample_loss, domain_ids):
        device = per_sample_loss.device
        domain_losses = torch.zeros(self.n_domains, device=device, dtype=torch.float32)
        present = torch.zeros(self.n_domains, dtype=torch.bool, device=device)

        for d in range(self.n_domains):
            mask = (domain_ids == d)
            if mask.any():
                domain_losses[d] = per_sample_loss[mask].float().mean()
                present[d] = True

        if not present.any():
            return per_sample_loss.mean()

        if self.training:
            with torch.no_grad():
                max_l = domain_losses.detach().max()
                updated_q = self.q_weights * torch.exp(self.step_size * torch.clamp(domain_losses.detach() - max_l, -5.0, 0.0))
                self.q_weights.copy_(updated_q / (updated_q.sum() + 1e-8))

        active_q = self.q_weights[present]
        active_q = active_q / (active_q.sum() + 1e-8)
        return (active_q * domain_losses[present]).sum()

def compute_onh_localization_loss(stn_tensor, saliency_prior, s_min=0.20, s_max=0.45, lambda_s=2.0):
    if isinstance(stn_tensor, (tuple, list)):
        cx, cy, s = stn_tensor[0].float(), stn_tensor[1].float(), stn_tensor[2].float()
    else:
        cx, cy, s = stn_tensor[:, 0].float(), stn_tensor[:, 1].float(), stn_tensor[:, 2].float()
    B = cx.shape[0]
    grid = torch.stack([cx, cy], dim=-1).view(B, 1, 1, 2)
    sampled_prior = F.grid_sample(
        saliency_prior.float().unsqueeze(1), grid, mode="bilinear", padding_mode="border", align_corners=False
    ).view(-1)
    uniform_baseline = 1.0 / float(saliency_prior.shape[-1] * saliency_prior.shape[-2])
    loss_prior = F.relu(-torch.log(torch.clamp(sampled_prior / uniform_baseline, min=1e-3))).mean()
    loss_scale = (F.relu(s - s_max).pow(2) + F.relu(s_min - s).pow(2)).mean()
    return loss_prior + lambda_s * loss_scale

class GlaucoGateTotalLoss(nn.Module):
    def __init__(self, loss_type="memory_pauc_bce", target_coverage=0.90,
                 lambda_q=0.25, lambda_loc=0.05, lambda_cov=2.0, lambda_dro=0.10,
                 supervise_quality=True, use_group_dro=True):
        super().__init__()
        self.loss_type = loss_type
        self.target_coverage = target_coverage
        self.lambda_q = lambda_q if supervise_quality else 0.0
        self.lambda_loc = lambda_loc
        self.lambda_cov = lambda_cov
        self.lambda_dro = lambda_dro if use_group_dro else 0.0
        self.pauc_mem = MemoryBankPartialAUCLoss(use_memory=(loss_type == "memory_pauc_bce"))
        self.group_dro = PseudoDomainGroupDRO(n_domains=10)

    def compute_cls_loss_per_sample(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits.float(), targets.float(), reduction="none")
        if self.loss_type == "focal":
            pt = torch.exp(-bce)
            return ((1.0 - pt) ** 2.0) * bce
        elif self.loss_type == "bce_hnm":
            neg_mask = (targets < 0.5)
            if neg_mask.sum() > 2:
                thresh = torch.quantile(bce[neg_mask].detach(), 0.75)
                weight = torch.where(neg_mask & (bce < thresh), torch.full_like(bce, 0.25), torch.ones_like(bce))
                return bce * weight
        return bce

    def forward(self, outputs, batch):
        with torch.amp.autocast("cuda" if outputs["logit"].is_cuda else "cpu", enabled=False):
            logits = outputs["logit"].float()
            targets = batch["label"].float()
            c_hat = torch.clamp(outputs["selection_score"].float(), 0.50, 1.0)

            per_sample_bce = self.compute_cls_loss_per_sample(logits, targets)
            base_bce = per_sample_bce.mean()

            mean_cov_det = torch.clamp(c_hat.mean().detach(), min=0.50)
            selective_bce = (c_hat.detach() * per_sample_bce).mean() / mean_cov_det

            sel_align = ((c_hat - (1.0 - torch.clamp(per_sample_bce.detach(), 0.0, 1.0))).pow(2)).mean()
            cov_penalty = self.lambda_cov * F.relu(self.target_coverage - c_hat.mean()).pow(2)
            loss_sel = 0.5 * base_bce + 0.5 * selective_bce + 0.1 * sel_align + cov_penalty

            if self.loss_type in ["minibatch_pauc", "memory_pauc_bce"]:
                loss_pauc = self.pauc_mem(logits, targets)
                loss_cls_total = 0.70 * loss_sel + 0.30 * loss_pauc
            else:
                loss_cls_total = loss_sel

            loss_dro = torch.tensor(0.0, device=logits.device)
            if self.lambda_dro > 0 and "pseudo_domain" in batch:
                loss_dro = self.group_dro(per_sample_bce, batch["pseudo_domain"])

            loss_q = torch.tensor(0.0, device=logits.device)
            if "m_q_logit" in outputs and self.lambda_q > 0:
                m_q_logit_pred = F.interpolate(
                    outputs["m_q_logit"].float(), size=batch["gt_qmap"].shape[-2:], mode="bilinear", align_corners=False
                )
                loss_q = F.binary_cross_entropy_with_logits(m_q_logit_pred, batch["gt_qmap"].float())

            loss_loc = torch.tensor(0.0, device=logits.device)
            if "stn_params" in outputs and self.lambda_loc > 0:
                loss_loc = compute_onh_localization_loss(outputs["stn_params"], batch["saliency_prior"])

            total = loss_cls_total + self.lambda_q * loss_q + self.lambda_loc * loss_loc + self.lambda_dro * loss_dro
            return total, {
                "loss_total": total.item(),
                "loss_cls": loss_cls_total.item(),
                "loss_q": loss_q.item(),
                "loss_loc": loss_loc.item(),
                "loss_dro": loss_dro.item()
            }